In [ ]:
import sys
sys.path.insert(0, 'src')

from fake_base_station.ng import (
    NGSetupRequestConfig,
    encode_ngsetup_request,
    decode_ngsetup_request,
    print_decoded_structure,
    PCAPTrafficReplayer
)

In [ ]:
# Create configuration with default values
config = NGSetupRequestConfig()


In [ ]:
# Encode the NGSetupRequest message
encoded = encode_ngsetup_request(config)

print("Encoded hex:")
print(encoded.hex())

# Decode back for verification
pdu2 = decode_ngsetup_request(encoded)
print("\nDecoded structure:")
print_decoded_structure(pdu2)


In [ ]:
from pycrate_mobile.NAS import *

Msg, err = parse_NAS_MO(unhexlify(encoded.hex()))

show(Msg)


## Send packets from file

In [ ]:
# Load in pcap file
traffic_replayer = PCAPTrafficReplayer(pcap_file="./test/gnb_ngap.pcap")
traffic_replayer.summary()

In [ ]:
from fake_base_station.ng import create_sctp_socket, NGAP_SCTP_PORT

HOST = "127.0.0.1"
PORT = NGAP_SCTP_PORT  # 38412 - standard NGAP/SCTP port per 3GPP TS 38.412

# Option 1: blocking replay over SCTP (opens/closes its own socket)
# traffic_replayer.replay_to_sctp(HOST, PORT, speed_factor=1.0)

# Option 2: background thread replay over SCTP
thread = traffic_replayer.replay_sctp_threaded(HOST, PORT, speed_factor=1.0, packet_type="ngap_only")

print(f"SCTP replay started in background (thread alive: {thread.is_alive()})")

# To stop early:
# thread.stop_event.set()

# To wait for completion:
# thread.join()

# Option 3: bring your own SCTP socket (e.g. to reuse an existing connection)
# sock = create_sctp_socket(HOST, PORT)
# thread = traffic_replayer.replay_threaded(sock, speed_factor=1.0)
# thread.join()
# sock.close()
